# 🚀 프로젝트: KoChatGPT 업그레이드하기

**AIFFEL LLM Trend Note2 — 프로젝트 노드**

## 📋 루브릭 체크리스트

| # | 평가기준 | 구현 위치 |
|---|---|---|
| ① | 데이터 정제 + Generation 기법 실험으로 정량적 성능 향상 | Section 2, 7 |
| ② | 새로운 데이터(KorQuAD) 수집·전처리로 성능 향상 | Section 3 |
| ③ | SFT/RM 학습 전략 적용 | Section 4, 5 |
| ④ | **SFT 모델 vs RM 모델** 정량/정성 비교 | Section 6-B |
| ⑤ | **KoGPT2 기본 vs SFT 모델** 정량/정성 비교 | Section 6-A |

---

## 0. 환경 설치 및 라이브러리 로드

In [ ]:
# 필요 패키지 설치
!pip install datasets loralib trl rouge-score nltk accelerate -q
!pip install transformers>=4.30.0 -q

# KoChatGPT 소스 클론
import os
if not os.path.exists('/content/KoChatGPT'):
    !git clone https://github.com/airobotlab/KoChatGPT
    !cp -r /content/KoChatGPT/colossalai_ChatGPT_230319/chatgpt /content/chatgpt
    print('KoChatGPT 클론 완료')
else:
    print('이미 클론되어 있습니다')

In [ ]:
import os, json, re, sys, copy, random, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '/content')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

import torch
import torch.nn as nn
from torch.utils.data import Dataset as TorchDataset, DataLoader

from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer,
    set_seed
)
from datasets import load_dataset

import nltk
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
nltk.download('punkt', quiet=True)

# ── 시드 고정 ─────────────────────────────────────────────────────────────────
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

---
## Section 1: 기존 데이터셋 EDA

In [ ]:
DATA_DIR = '/content/KoChatGPT/data_kochatgpt'

def load_jsonl(path):
    data = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                data.append(json.loads(line))
    return data

sft_raw = load_jsonl(os.path.join(DATA_DIR, 'kochatgpt_1_SFT.jsonl'))
rm_raw  = load_jsonl(os.path.join(DATA_DIR, 'kochatgpt_2_RM.jsonl'))
ppo_raw = load_jsonl(os.path.join(DATA_DIR, 'kochatgpt_3_PPO.jsonl'))

print(f'SFT: {len(sft_raw):,}개 | RM: {len(rm_raw):,}개 | PPO: {len(ppo_raw):,}개')
print('\n▶ SFT 샘플 (1번째):')
print(json.dumps(sft_raw[0], ensure_ascii=False, indent=2))

In [ ]:
# ── 길이 분포 EDA ─────────────────────────────────────────────────────────────
sft_df = pd.DataFrame(sft_raw)
sft_df['prompt_len']     = sft_df['prompt'].apply(lambda x: len(str(x)))
sft_df['completion_len'] = sft_df['completion'].apply(lambda x: len(str(x)))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('SFT 데이터셋 EDA', fontsize=13, fontweight='bold')

axes[0].hist(sft_df['prompt_len'], bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(sft_df['prompt_len'].median(), color='red', ls='--', lw=1.5, label='중앙값')
axes[0].set_title('Prompt 길이 분포'); axes[0].set_xlabel('문자 수'); axes[0].legend()

axes[1].hist(sft_df['completion_len'], bins=40, color='tomato', edgecolor='white')
axes[1].axvline(sft_df['completion_len'].median(), color='navy', ls='--', lw=1.5, label='중앙값')
axes[1].set_title('Completion 길이 분포'); axes[1].set_xlabel('문자 수'); axes[1].legend()

axes[2].scatter(sft_df['prompt_len'], sft_df['completion_len'],
                alpha=0.3, s=10, color='mediumseagreen')
axes[2].set_title('Prompt vs Completion 길이')
axes[2].set_xlabel('Prompt'); axes[2].set_ylabel('Completion')

plt.tight_layout()
plt.savefig('/content/eda_length.png', dpi=150, bbox_inches='tight')
plt.show()
print(sft_df[['prompt_len','completion_len']].describe().round(1))

In [ ]:
# ── 품질 이슈 탐지 ────────────────────────────────────────────────────────────
def detect_issues(text):
    issues = []
    t = str(text)
    if not t.strip():                                       issues.append('빈 텍스트')
    if re.search(r'[\u4e00-\u9fff]{3,}', t):               issues.append('한자 포함')
    if len(t) < 5:                                          issues.append('너무 짧음')
    if len(t) > 1000:                                       issues.append('너무 김')
    tokens = t.split()
    if len(tokens) > 5 and len(set(tokens))/len(tokens) < 0.4:
                                                            issues.append('반복 토큰')
    return issues

sft_df['comp_issues'] = sft_df['completion'].apply(detect_issues)
has_issue = sft_df['comp_issues'].apply(lambda x: len(x) > 0)
all_issues = [i for lst in sft_df['comp_issues'] for i in lst]
issue_ser  = pd.Series(all_issues).value_counts()

print('=== Completion 품질 이슈 현황 ===')
print(issue_ser.to_string() if not issue_ser.empty else '이슈 없음')
print(f'\n이슈 있는 샘플: {has_issue.sum():,} / {len(sft_df):,}  ({has_issue.mean()*100:.1f}%)')

if not issue_ser.empty:
    plt.figure(figsize=(7,3))
    issue_ser.plot(kind='bar', color='salmon', edgecolor='white')
    plt.title('Completion 이슈 유형별 빈도'); plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig('/content/eda_issues.png', dpi=150, bbox_inches='tight')
    plt.show()

---
## Section 2: 데이터 정제 (Cleaning)

In [ ]:
def clean_text(text):
    if not isinstance(text, str): return ''
    text = re.sub(r'[\u4e00-\u9fff]+', '', text)     # 한자 제거
    text = re.sub(r'[\x00-\x1f\x7f]', '', text)      # 제어문자 제거
    text = re.sub(r'[ \t]+', ' ', text).strip()       # 연속 공백 정규화
    return text

def is_valid(p, c, min_p=5, max_p=400, min_c=10, max_c=900):
    p, c = clean_text(p), clean_text(c)
    if not (min_p <= len(p) <= max_p): return False
    if not (min_c <= len(c) <= max_c): return False
    tokens = c.split()
    if len(tokens) > 5 and len(set(tokens))/len(tokens) < 0.4: return False
    return True

# SFT 정제
cleaned_sft = []
for item in sft_raw:
    p = clean_text(item.get('prompt', ''))
    c = clean_text(item.get('completion', ''))
    if is_valid(p, c):
        cleaned_sft.append({'prompt': p, 'completion': c})

# RM 정제
cleaned_rm = []
for item in rm_raw:
    p  = clean_text(item.get('prompt', ''))
    c0 = clean_text(item.get('completion_0', ''))
    c1 = clean_text(item.get('completion_1', ''))
    c2 = clean_text(item.get('completion_2', ''))
    if all(is_valid(p, c, min_c=5) for c in [c0, c1, c2]):
        cleaned_rm.append({'prompt': p,
                           'completion_0': c0,
                           'completion_1': c1,
                           'completion_2': c2})

print(f'SFT  전: {len(sft_raw):,} → 후: {len(cleaned_sft):,}  '
      f'(제거 {len(sft_raw)-len(cleaned_sft):,}개, {(1-len(cleaned_sft)/len(sft_raw))*100:.1f}%)')
print(f'RM   전: {len(rm_raw):,} → 후: {len(cleaned_rm):,}  '
      f'(제거 {len(rm_raw)-len(cleaned_rm):,}개, {(1-len(cleaned_rm)/len(rm_raw))*100:.1f}%)')

---
## Section 3: 새로운 데이터 수집 — KorQuAD 1.0

In [ ]:
# KorQuAD 1.0 로드
korquad = load_dataset('squad_kor_v1', split='train')
print(f'KorQuAD 원본: {len(korquad):,}개')
print(f'샘플: Q={korquad[0]["question"]}  A={korquad[0]["answers"]["text"]}')

In [ ]:
def korquad_to_sft(example, max_ctx=150):
    """
    KorQuAD → SFT 형식 변환
    prompt:     '질문: {question}'
    completion: '{answer}. (근거: {context 앞부분})'
    """
    q = clean_text(example['question'])
    answers = example['answers']['text']
    if not answers: return None
    a   = clean_text(answers[0])
    ctx = clean_text(example['context'])[:max_ctx]

    prompt     = f'질문: {q}'
    completion = f'{a}. (근거: {ctx})'
    if is_valid(prompt, completion, min_c=8, max_c=700):
        return {'prompt': prompt, 'completion': completion}
    return None

korquad_sft = [r for ex in korquad if (r := korquad_to_sft(ex))]
random.shuffle(korquad_sft)
korquad_sft_sampled = korquad_sft[:len(cleaned_sft)]   # 원본과 1:1 비율

augmented_sft = cleaned_sft + korquad_sft_sampled
random.shuffle(augmented_sft)

print(f'정제된 원본 SFT : {len(cleaned_sft):,}개')
print(f'KorQuAD 추가분  : {len(korquad_sft_sampled):,}개')
print(f'최종 SFT 합계   : {len(augmented_sft):,}개')
print('\nKorQuAD 변환 샘플:')
print(json.dumps(korquad_sft_sampled[0], ensure_ascii=False, indent=2))

In [ ]:
# 정제·증량 데이터 저장
SAVE_DIR = '/content/data_cleaned'
os.makedirs(SAVE_DIR, exist_ok=True)

def save_jsonl(data, path):
    with open(path, 'w', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f'  저장: {path}  ({len(data):,}개)')

save_jsonl(augmented_sft, os.path.join(SAVE_DIR, 'augmented_sft.jsonl'))
save_jsonl(cleaned_rm,    os.path.join(SAVE_DIR, 'cleaned_rm.jsonl'))
save_jsonl(ppo_raw,       os.path.join(SAVE_DIR, 'ppo.jsonl'))

---
## Section 4: Supervised Fine-Tuning (SFT)

**Foundation Model: `skt/kogpt2-base-v2`**

In [ ]:
MODEL_NAME = 'skt/kogpt2-base-v2'

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    bos_token='</s>', eos_token='</s>',
    unk_token='<unk>', pad_token='<pad>'
)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
base_model.config.pad_token_id = tokenizer.pad_token_id
print(f'{MODEL_NAME} 로드 완료')
print(f'파라미터 수: {sum(p.numel() for p in base_model.parameters()):,}')

In [ ]:
class SFTDataset(TorchDataset):
    """
    <bos> 아래 질문에 답하세요.\n{prompt}\n\n{completion} <eos>
    형식으로 인코딩 후 CLM 방식으로 전체 시퀀스에 대해 loss 계산
    """
    def __init__(self, data, tokenizer, max_len=512):
        self.data = data; self.tokenizer = tokenizer; self.max_len = max_len

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        text = (f"{self.tokenizer.bos_token}"
                f"아래 질문에 답하세요.\n{item['prompt']}\n\n"
                f"{item['completion']}"
                f"{self.tokenizer.eos_token}")
        enc = self.tokenizer(
            text, max_length=self.max_len, truncation=True,
            padding='max_length', return_tensors='pt'
        )
        ids   = enc['input_ids'].squeeze()
        mask  = enc['attention_mask'].squeeze()
        labels = ids.clone(); labels[mask == 0] = -100
        return {'input_ids': ids, 'attention_mask': mask, 'labels': labels}

split_idx = int(len(augmented_sft) * 0.9)
train_ds  = SFTDataset(augmented_sft[:split_idx], tokenizer)
val_ds    = SFTDataset(augmented_sft[split_idx:], tokenizer)
print(f'학습: {len(train_ds):,}개 | 검증: {len(val_ds):,}개')

In [ ]:
SFT_OUTPUT = '/content/model_sft'

sft_args = TrainingArguments(
    output_dir=SFT_OUTPUT,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,    # 실효 배치 16
    learning_rate=5e-5,
    lr_scheduler_type='cosine',
    warmup_steps=100,
    weight_decay=0.01,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    logging_steps=50,
    fp16=torch.cuda.is_available(),
    report_to='none',
    seed=SEED
)

sft_model   = copy.deepcopy(base_model)
sft_trainer = Trainer(
    model=sft_model,
    args=sft_args,
    train_dataset=train_ds,
    eval_dataset=val_ds
)

print('SFT 학습 시작...')
sft_result = sft_trainer.train()
sft_trainer.save_model(SFT_OUTPUT)
tokenizer.save_pretrained(SFT_OUTPUT)
print(f'SFT 완료 | train_loss: {sft_result.training_loss:.4f}')

---
## Section 5: Reward Model (RM) 학습

In [ ]:
class RewardModel(nn.Module):
    """
    GPT2 위에 value_head (Linear hidden→1) 를 추가한 Reward Model
    마지막 유효 토큰의 hidden state → scalar reward
    """
    def __init__(self, pretrained):
        super().__init__()
        self.model      = pretrained
        self.value_head = nn.Linear(self.model.config.n_embd, 1, bias=False)
        nn.init.normal_(self.value_head.weight, std=0.02)

    def forward(self, input_ids, attention_mask=None):
        out    = self.model(input_ids=input_ids, attention_mask=attention_mask,
                            output_hidden_states=True)
        hidden = out.hidden_states[-1]              # (B, T, H)
        if attention_mask is not None:
            seq_lens = attention_mask.sum(dim=1) - 1
            bidx     = torch.arange(hidden.size(0), device=hidden.device)
            pooled   = hidden[bidx, seq_lens]       # (B, H)
        else:
            pooled = hidden[:, -1, :]
        return self.value_head(pooled).squeeze(-1)  # (B,)

sft_backbone = AutoModelForCausalLM.from_pretrained(SFT_OUTPUT)
reward_model = RewardModel(sft_backbone).to(DEVICE)
print(f'RewardModel 파라미터: {sum(p.numel() for p in reward_model.parameters()):,}')

In [ ]:
class RMDataset(TorchDataset):
    """
    completion_0 > completion_1 > completion_2 품질 가정
    가능한 모든 (chosen, rejected) 쌍 생성
    """
    def __init__(self, data, tokenizer, max_len=512):
        self.tokenizer = tokenizer; self.max_len = max_len
        self.pairs = []
        for item in data:
            p  = item['prompt']
            c0, c1, c2 = item['completion_0'], item['completion_1'], item['completion_2']
            for ch, rj in [(c0,c1),(c0,c2),(c1,c2)]:
                self.pairs.append((p, ch, rj))

    def __len__(self): return len(self.pairs)

    def encode(self, prompt, completion):
        text = f'{self.tokenizer.bos_token}{prompt}\n{completion}{self.tokenizer.eos_token}'
        enc  = self.tokenizer(text, max_length=self.max_len, truncation=True,
                              padding='max_length', return_tensors='pt')
        return enc['input_ids'].squeeze(), enc['attention_mask'].squeeze()

    def __getitem__(self, idx):
        p, ch, rj = self.pairs[idx]
        c_ids, c_mask = self.encode(p, ch)
        r_ids, r_mask = self.encode(p, rj)
        return {'chosen_input_ids': c_ids, 'chosen_attention_mask': c_mask,
                'rejected_input_ids': r_ids, 'rejected_attention_mask': r_mask}

rm_split      = int(len(cleaned_rm) * 0.9)
rm_train_ds   = RMDataset(cleaned_rm[:rm_split], tokenizer)
rm_val_ds     = RMDataset(cleaned_rm[rm_split:], tokenizer)
print(f'RM 학습 쌍: {len(rm_train_ds):,} | 검증 쌍: {len(rm_val_ds):,}')

In [ ]:
def pairwise_loss(chosen_r, rejected_r):
    """Bradley-Terry 기반 pairwise ranking loss"""
    return -torch.mean(torch.log(torch.sigmoid(chosen_r - rejected_r)))

RM_OUTPUT = '/content/model_rm'
RM_EPOCHS = 2
RM_LR     = 2e-5
RM_BATCH  = 4

tr_loader = DataLoader(rm_train_ds, batch_size=RM_BATCH, shuffle=True)
vl_loader = DataLoader(rm_val_ds,   batch_size=RM_BATCH, shuffle=False)

optimizer = torch.optim.AdamW(reward_model.parameters(), lr=RM_LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=RM_EPOCHS * len(tr_loader)
)

rm_tr_losses, rm_vl_losses, rm_vl_accs = [], [], []

print('RM 학습 시작...')
for epoch in range(RM_EPOCHS):
    # ── Train ─────────────────────────────────────────────────────────────────
    reward_model.train()
    ep_loss = 0
    for step, batch in enumerate(tr_loader):
        c_ids  = batch['chosen_input_ids'].to(DEVICE)
        c_mask = batch['chosen_attention_mask'].to(DEVICE)
        r_ids  = batch['rejected_input_ids'].to(DEVICE)
        r_mask = batch['rejected_attention_mask'].to(DEVICE)

        ch_r = reward_model(c_ids, c_mask)
        rj_r = reward_model(r_ids, r_mask)
        loss = pairwise_loss(ch_r, rj_r)

        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(reward_model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        ep_loss += loss.item()

        if (step+1) % 50 == 0:
            print(f'  E{epoch+1} | step {step+1}/{len(tr_loader)} | loss={loss.item():.4f}')

    rm_tr_losses.append(ep_loss / len(tr_loader))

    # ── Validate ──────────────────────────────────────────────────────────────
    reward_model.eval()
    vl_loss = correct = total = 0
    with torch.no_grad():
        for batch in vl_loader:
            c_ids  = batch['chosen_input_ids'].to(DEVICE)
            c_mask = batch['chosen_attention_mask'].to(DEVICE)
            r_ids  = batch['rejected_input_ids'].to(DEVICE)
            r_mask = batch['rejected_attention_mask'].to(DEVICE)
            ch_r = reward_model(c_ids, c_mask)
            rj_r = reward_model(r_ids, r_mask)
            vl_loss += pairwise_loss(ch_r, rj_r).item()
            correct += (ch_r > rj_r).sum().item()
            total   += ch_r.size(0)

    avg_vl = vl_loss / len(vl_loader)
    acc    = correct / total
    rm_vl_losses.append(avg_vl); rm_vl_accs.append(acc)
    print(f'\nEpoch {epoch+1}/{RM_EPOCHS} | tr_loss={rm_tr_losses[-1]:.4f} '
          f'| vl_loss={avg_vl:.4f} | val_acc={acc*100:.1f}%\n')

os.makedirs(RM_OUTPUT, exist_ok=True)
torch.save(reward_model.state_dict(), os.path.join(RM_OUTPUT, 'rm_weights.pt'))
print('RM 저장 완료')

In [ ]:
# RM 학습 곡선 시각화
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
fig.suptitle('Reward Model 학습 결과', fontsize=13, fontweight='bold')

ep = range(1, RM_EPOCHS+1)
ax1.plot(ep, rm_tr_losses, 'o-', label='Train Loss', color='steelblue')
ax1.plot(ep, rm_vl_losses, 's-', label='Val Loss',   color='tomato')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.legend()
ax1.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

ax2.plot(ep, [a*100 for a in rm_vl_accs], 'D-', color='mediumseagreen', ms=8)
ax2.axhline(50, color='gray', ls='--', lw=1, label='Random Baseline (50%)')
ax2.set_title('Val Accuracy (chosen > rejected)')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('%'); ax2.legend()
ax2.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

plt.tight_layout()
plt.savefig('/content/rm_training_curve.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 6: 모델 비교 분석 (정량 + 정성)

### 공통 도구 정의

In [ ]:
# ── 생성 함수 ─────────────────────────────────────────────────────────────────
def generate(model, tokenizer, prompt, strategy='beam', max_new=150):
    """
    strategy: 'greedy' | 'beam' | 'top_k' | 'top_p'
    """
    instruction = f"{tokenizer.bos_token}아래 질문에 답하세요.\n{prompt}\n\n"
    enc = tokenizer(instruction, return_tensors='pt').to(DEVICE)
    model.eval()

    kwargs = dict(
        input_ids=enc['input_ids'],
        attention_mask=enc['attention_mask'],
        max_new_tokens=max_new,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )
    if strategy == 'greedy':
        kwargs.update(do_sample=False)
    elif strategy == 'beam':
        kwargs.update(do_sample=False, num_beams=5, no_repeat_ngram_size=3, early_stopping=True)
    elif strategy == 'top_k':
        kwargs.update(do_sample=True, top_k=50, temperature=0.8)
    elif strategy == 'top_p':
        kwargs.update(do_sample=True, top_p=0.9, temperature=0.8)

    with torch.no_grad():
        out = model.generate(**kwargs)
    gen_ids = out[0][enc['input_ids'].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()


# ── 평가 메트릭 ───────────────────────────────────────────────────────────────
rouge_sc  = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=False)
smoother  = SmoothingFunction().method4

def calc_metrics(hyp, ref):
    ht, rt = hyp.split(), ref.split()
    if not ht:
        return {k: 0.0 for k in ['bleu1','bleu4','rouge1','rouge2','rougeL','unique_ratio','length']}
    b1 = sentence_bleu([rt], ht, weights=(1,0,0,0), smoothing_function=smoother)
    b4 = sentence_bleu([rt], ht, smoothing_function=smoother)
    rg = rouge_sc.score(ref, hyp)
    return {
        'bleu1':        round(b1, 4),
        'bleu4':        round(b4, 4),
        'rouge1':       round(rg['rouge1'].fmeasure, 4),
        'rouge2':       round(rg['rouge2'].fmeasure, 4),
        'rougeL':       round(rg['rougeL'].fmeasure, 4),
        'unique_ratio': round(len(set(ht))/len(ht), 4),
        'length':       len(ht)
    }

print('생성 함수 & 메트릭 정의 완료')

In [ ]:
# 테스트 샘플 (참조 답변 포함)
test_samples = [
    {'prompt': '인공지능이란 무엇인가요?',
     'reference': '인공지능은 인간의 학습 추론 문제해결 능력을 컴퓨터로 구현한 기술입니다.'},
    {'prompt': '기후변화의 주요 원인은 무엇인가요?',
     'reference': '기후변화의 주요 원인은 화석연료 연소와 산업 활동으로 발생하는 온실가스 증가입니다.'},
    {'prompt': '건강한 식단을 위해 어떤 음식을 먹어야 하나요?',
     'reference': '건강한 식단은 채소 과일 통곡물 단백질이 풍부한 식품을 균형 있게 섭취하는 것이 중요합니다.'},
    {'prompt': '한국의 전통 명절에 대해 설명해 주세요.',
     'reference': '한국의 대표 명절인 설날과 추석에는 가족이 모여 차례를 지내고 전통 음식을 나눕니다.'},
    {'prompt': '독서의 장점은 무엇인가요?',
     'reference': '독서는 어휘력과 지식을 넓히고 집중력과 창의력을 향상시키며 스트레스 해소에도 도움이 됩니다.'}
]

# 모델 준비
base_m = copy.deepcopy(base_model).to(DEVICE)
sft_m  = AutoModelForCausalLM.from_pretrained(SFT_OUTPUT).to(DEVICE)
print(f'테스트 샘플: {len(test_samples)}개  |  모델: base / sft 준비 완료')

### 6-A: KoGPT2 기본 모델 vs SFT 모델 비교

In [ ]:
print('='*68)
print('  [비교 A]  KoGPT2 기본  vs  SFT 모델  (Beam Search)')
print('='*68)

res_base, res_sft = [], []

for i, s in enumerate(test_samples):
    p, ref = s['prompt'], s['reference']
    out_b  = generate(base_m, tokenizer, p, 'beam')
    out_s  = generate(sft_m,  tokenizer, p, 'beam')
    m_b, m_s = calc_metrics(out_b, ref), calc_metrics(out_s, ref)
    res_base.append(m_b); res_sft.append(m_s)

    print(f'\n[{i+1}] {p}')
    print(f'  기본: {out_b[:110]}')
    print(f'        BLEU-4={m_b["bleu4"]:.4f} ROUGE-L={m_b["rougeL"]:.4f} unique={m_b["unique_ratio"]:.2f}')
    print(f'  SFT:  {out_s[:110]}')
    print(f'        BLEU-4={m_s["bleu4"]:.4f} ROUGE-L={m_s["rougeL"]:.4f} unique={m_s["unique_ratio"]:.2f}')

In [ ]:
# ── 정량 요약 테이블 ─────────────────────────────────────────────────────────
mk = ['bleu1','bleu4','rouge1','rouge2','rougeL','unique_ratio']
avg_base = {k: np.mean([r[k] for r in res_base]) for k in mk}
avg_sft  = {k: np.mean([r[k] for r in res_sft])  for k in mk}

summary_A = pd.DataFrame({'KoGPT2 기본': avg_base, 'SFT 모델': avg_sft}).T
print('\n평균 정량 지표 (KoGPT2 기본 vs SFT)')
print(summary_A.to_string(float_format='%.4f'))

print('\n개선율:')
for k in ['bleu4','rougeL','unique_ratio']:
    d   = avg_sft[k] - avg_base[k]
    pct = d / (avg_base[k] + 1e-8) * 100
    print(f'  {k:14s}: {pct:+.1f}%  ({avg_base[k]:.4f} → {avg_sft[k]:.4f})')

In [ ]:
# 시각화 A
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('KoGPT2 기본 vs SFT 모델 정량 비교', fontsize=13, fontweight='bold')

x     = np.arange(len(mk))
w     = 0.35
b_v   = [avg_base[k] for k in mk]
s_v   = [avg_sft[k]  for k in mk]
axes[0].bar(x-w/2, b_v, w, label='KoGPT2 기본', color='steelblue', edgecolor='white')
axes[0].bar(x+w/2, s_v, w, label='SFT 모델',    color='tomato',    edgecolor='white')
axes[0].set_xticks(x); axes[0].set_xticklabels(mk, rotation=15, fontsize=8)
axes[0].set_title('평균 메트릭'); axes[0].legend()

impr = [(avg_sft[k]-avg_base[k])/(avg_base[k]+1e-8)*100 for k in mk]
clrs = ['mediumseagreen' if v >= 0 else 'salmon' for v in impr]
axes[1].barh(mk, impr, color=clrs, edgecolor='white')
axes[1].axvline(0, color='black', lw=0.8)
axes[1].set_title('SFT 적용 후 개선율 (%)'); axes[1].set_xlabel('개선율 (%)')

plt.tight_layout()
plt.savefig('/content/compare_base_vs_sft.png', dpi=150, bbox_inches='tight')
plt.show()

### 6-B: SFT 모델 vs Reward Model 스코어링 비교

In [ ]:
print('='*68)
print('  [비교 B]  SFT 모델  vs  RM 스코어링으로 후보 선택')
print('='*68)

strategies = ['greedy','beam','top_k','top_p']
reward_model.eval(); sft_m.eval()

rm_comparison = []
for i, s in enumerate(test_samples):
    p, ref = s['prompt'], s['reference']

    # 전략별 후보 생성
    cands = {st: generate(sft_m, tokenizer, p, st) for st in strategies}

    # RM 점수 계산
    rm_scores = {}
    for st, cand in cands.items():
        text = f"{tokenizer.bos_token}{p}\n{cand}{tokenizer.eos_token}"
        enc  = tokenizer(text, return_tensors='pt', max_length=512,
                         truncation=True, padding='max_length').to(DEVICE)
        with torch.no_grad():
            rm_scores[st] = reward_model(enc['input_ids'],
                                          enc['attention_mask']).item()

    best_st  = max(rm_scores, key=rm_scores.get)
    best_out = cands[best_st]
    m_best   = calc_metrics(best_out, ref)
    # greedy를 SFT baseline으로
    m_greedy = calc_metrics(cands['greedy'], ref)

    rm_comparison.append({'prompt': p, 'best_strategy': best_st,
                           'best_output': best_out, 'rm_score': rm_scores[best_st],
                           'metrics_best': m_best, 'metrics_greedy': m_greedy,
                           'rm_scores': rm_scores, 'cands': cands})

    print(f'\n[{i+1}] {p}')
    for st, sc in rm_scores.items():
        mark = ' ← RM 선택' if st == best_st else ''
        print(f'  {st:8s}: RM={sc:+.4f}{mark}')
    print(f'  선택 출력: {best_out[:100]}')
    print(f'  SFT(greedy) BLEU-4={m_greedy["bleu4"]:.4f}  '
          f'→  RM-선택 BLEU-4={m_best["bleu4"]:.4f}')

In [ ]:
# ── SFT vs RM 정량 비교 요약 ──────────────────────────────────────────────────
avg_sft_g  = {k: np.mean([c['metrics_greedy'][k] for c in rm_comparison]) for k in mk}
avg_rm_sel = {k: np.mean([c['metrics_best'][k]   for c in rm_comparison]) for k in mk}

summary_B = pd.DataFrame({'SFT(greedy)': avg_sft_g, 'RM 선택': avg_rm_sel}).T
print('\n평균 정량 지표 (SFT Greedy vs RM 스코어 선택)')
print(summary_B.to_string(float_format='%.4f'))

print('\n개선율:')
for k in ['bleu4','rougeL']:
    d   = avg_rm_sel[k] - avg_sft_g[k]
    pct = d / (avg_sft_g[k] + 1e-8) * 100
    print(f'  {k:10s}: {pct:+.1f}%  ({avg_sft_g[k]:.4f} → {avg_rm_sel[k]:.4f})')

In [ ]:
# 시각화 B
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('SFT 모델 vs RM 스코어 선택 비교', fontsize=13, fontweight='bold')

# 전략별 평균 RM 점수
avg_rm_by_st = {st: np.mean([c['rm_scores'][st] for c in rm_comparison])
                for st in strategies}
bars = axes[0].bar(avg_rm_by_st.keys(), avg_rm_by_st.values(),
                   color=['steelblue','tomato','mediumseagreen','orchid'],
                   edgecolor='white')
axes[0].axhline(0, color='gray', ls='--', lw=0.8)
axes[0].set_title('전략별 평균 RM Score')
axes[0].set_ylabel('RM Score')
for bar, val in zip(bars, avg_rm_by_st.values()):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.003,
                 f'{val:+.3f}', ha='center', va='bottom', fontsize=9)

# BLEU-4 / ROUGE-L: SFT greedy vs RM 선택
compare_keys = ['bleu4','rougeL']
x2 = np.arange(len(compare_keys)); w2 = 0.3
axes[1].bar(x2-w2/2, [avg_sft_g[k]  for k in compare_keys], w2,
            label='SFT(greedy)', color='steelblue', edgecolor='white')
axes[1].bar(x2+w2/2, [avg_rm_sel[k] for k in compare_keys], w2,
            label='RM 선택',     color='tomato',    edgecolor='white')
axes[1].set_xticks(x2); axes[1].set_xticklabels(compare_keys)
axes[1].set_title('SFT vs RM 선택 메트릭'); axes[1].legend()

plt.tight_layout()
plt.savefig('/content/compare_sft_vs_rm.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Section 7: 디코딩 하이퍼파라미터 실험

Beam size, Top-K, Temperature를 체계적으로 비교하여 최적 설정을 도출합니다.

In [ ]:
def gen_custom(model, tokenizer, prompt, max_new=150, **kwargs):
    instruction = f"{tokenizer.bos_token}아래 질문에 답하세요.\n{prompt}\n\n"
    enc = tokenizer(instruction, return_tensors='pt').to(DEVICE)
    model.eval()
    with torch.no_grad():
        out = model.generate(
            input_ids=enc['input_ids'],
            attention_mask=enc['attention_mask'],
            max_new_tokens=max_new,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            **kwargs
        )
    gen_ids = out[0][enc['input_ids'].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

decode_configs = [
    ('Greedy',         dict(do_sample=False)),
    ('Beam-3',         dict(do_sample=False, num_beams=3, no_repeat_ngram_size=3)),
    ('Beam-5',         dict(do_sample=False, num_beams=5, no_repeat_ngram_size=3)),
    ('Beam-10',        dict(do_sample=False, num_beams=10, no_repeat_ngram_size=3)),
    ('Top-K30 T0.7',   dict(do_sample=True, top_k=30,  temperature=0.7)),
    ('Top-K50 T0.9',   dict(do_sample=True, top_k=50,  temperature=0.9)),
    ('Top-P0.85',      dict(do_sample=True, top_p=0.85, temperature=0.8)),
    ('Top-P0.95',      dict(do_sample=True, top_p=0.95, temperature=0.9)),
]

prompts = [s['prompt']    for s in test_samples]
refs    = [s['reference'] for s in test_samples]

decode_results = []
for name, cfg in decode_configs:
    b4_list, rL_list, uniq_list = [], [], []
    for p, r in zip(prompts, refs):
        out = gen_custom(sft_m, tokenizer, p, **cfg)
        m   = calc_metrics(out, r)
        b4_list.append(m['bleu4']); rL_list.append(m['rougeL']); uniq_list.append(m['unique_ratio'])
    decode_results.append({'strategy': name,
                           'avg_bleu4':  np.mean(b4_list),
                           'avg_rougeL': np.mean(rL_list),
                           'avg_unique': np.mean(uniq_list)})
    print(f'  {name:18s} | BLEU-4={np.mean(b4_list):.4f}  '
          f'ROUGE-L={np.mean(rL_list):.4f}  unique={np.mean(uniq_list):.2f}')

decode_df = pd.DataFrame(decode_results).set_index('strategy')
print('\n' + decode_df.to_string(float_format='%.4f'))

In [ ]:
# 디코딩 전략 시각화
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('디코딩 전략별 성능 비교', fontsize=13, fontweight='bold')

pal = plt.cm.tab10.colors
strats = decode_df.index.tolist()
x = np.arange(len(strats))

for ax, col, title in zip(axes,
                          ['avg_bleu4','avg_rougeL','avg_unique'],
                          ['BLEU-4','ROUGE-L','Unique Ratio']):
    vals = decode_df[col].values
    best = np.argmax(vals)
    clrs = ['#55A868' if i == best else '#4C72B0' for i in range(len(vals))]
    ax.bar(x, vals, color=clrs, edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels(strats, rotation=30, ha='right', fontsize=7)
    ax.set_title(title)
    ax.text(best, vals[best] + 0.001, '★', ha='center', va='bottom',
            fontsize=12, color='gold')

plt.tight_layout()
plt.savefig('/content/decoding_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

best_st = decode_df['avg_rougeL'].idxmax()
print(f'ROUGE-L 기준 최적 전략: "{best_st}"')
print(f'  BLEU-4={decode_df.loc[best_st,"avg_bleu4"]:.4f}  '
      f'ROUGE-L={decode_df.loc[best_st,"avg_rougeL"]:.4f}')

---
## Section 8: 정성 평가 — 전체 모델 출력 비교표

In [ ]:
best_decode_st = decode_df['avg_rougeL'].idxmax()   # 최적 전략
best_cfg = dict(decode_configs[[n for n,_ in decode_configs].index(best_decode_st)][1])

print('='*70)
print('  정성 평가: 기본 모델 / SFT / RM 선택 출력 비교')
print('='*70)

for i, s in enumerate(test_samples):
    p, ref = s['prompt'], s['reference']
    out_b  = generate(base_m, tokenizer, p, 'beam')
    out_s  = gen_custom(sft_m, tokenizer, p, **best_cfg)

    # RM 선택
    cands_rm = {st: generate(sft_m, tokenizer, p, st) for st in strategies}
    rm_sc    = {}
    for st, cand in cands_rm.items():
        text = f"{tokenizer.bos_token}{p}\n{cand}{tokenizer.eos_token}"
        enc  = tokenizer(text, return_tensors='pt', max_length=512,
                         truncation=True, padding='max_length').to(DEVICE)
        with torch.no_grad():
            rm_sc[st] = reward_model(enc['input_ids'], enc['attention_mask']).item()
    out_rm = cands_rm[max(rm_sc, key=rm_sc.get)]

    m_b = calc_metrics(out_b, ref)
    m_s = calc_metrics(out_s, ref)
    m_r = calc_metrics(out_rm, ref)

    print(f'\n{"-"*70}')
    print(f'질문 {i+1}: {p}')
    print(f'{"-"*70}')
    print(f'[참조]      {ref}')
    print(f'[기본모델]  {out_b[:120]}')
    print(f'            BLEU-4={m_b["bleu4"]:.4f}  ROUGE-L={m_b["rougeL"]:.4f}  '
          f'unique={m_b["unique_ratio"]:.2f}')
    print(f'[SFT({best_decode_st})]  {out_s[:120]}')
    print(f'            BLEU-4={m_s["bleu4"]:.4f}  ROUGE-L={m_s["rougeL"]:.4f}  '
          f'unique={m_s["unique_ratio"]:.2f}')
    print(f'[RM 선택]   {out_rm[:120]}')
    print(f'            BLEU-4={m_r["bleu4"]:.4f}  ROUGE-L={m_r["rougeL"]:.4f}  '
          f'unique={m_r["unique_ratio"]:.2f}')

print(f'\n{"="*70}')
print('정성 평가 완료')

---
## Section 9: 최종 종합 대시보드

In [ ]:
fig = plt.figure(figsize=(18, 10))
fig.suptitle('KoChatGPT 업그레이드 — 최종 결과 대시보드', fontsize=15, fontweight='bold')

# ① 데이터셋 규모
ax1 = fig.add_subplot(2, 3, 1)
labels_d = ['원본 SFT', '정제 후', 'KorQuAD\n추가', '최종']
vals_d   = [len(sft_raw), len(cleaned_sft), len(korquad_sft_sampled), len(augmented_sft)]
b = ax1.bar(labels_d, vals_d, color=['steelblue','tomato','mediumseagreen','orchid'],
            edgecolor='white')
ax1.set_title('데이터셋 규모'); ax1.set_ylabel('샘플 수')
for bar, v in zip(b, vals_d):
    ax1.text(bar.get_x()+bar.get_width()/2, v+max(vals_d)*0.01,
             f'{v:,}', ha='center', va='bottom', fontsize=8)

# ② BLEU-4 비교
ax2 = fig.add_subplot(2, 3, 2)
mn = ['KoGPT2\n기본', 'SFT']
ax2.bar(mn, [avg_base['bleu4'], avg_sft['bleu4']],
        color=['steelblue','tomato'], edgecolor='white')
ax2.set_title('평균 BLEU-4'); ax2.set_ylabel('BLEU-4')

# ③ ROUGE-L 비교
ax3 = fig.add_subplot(2, 3, 3)
ax3.bar(mn, [avg_base['rougeL'], avg_sft['rougeL']],
        color=['steelblue','tomato'], edgecolor='white')
ax3.set_title('평균 ROUGE-L'); ax3.set_ylabel('ROUGE-L')

# ④ RM Validation Accuracy
ax4 = fig.add_subplot(2, 3, 4)
ax4.plot(range(1, RM_EPOCHS+1), [a*100 for a in rm_vl_accs],
         'D-', color='mediumseagreen', ms=9)
ax4.axhline(50, color='gray', ls='--', lw=1, label='Random (50%)')
ax4.set_title('RM Val Accuracy'); ax4.set_xlabel('Epoch'); ax4.set_ylabel('%')
ax4.legend(); ax4.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))

# ⑤ 디코딩 전략별 ROUGE-L
ax5 = fig.add_subplot(2, 3, 5)
rL_v  = decode_df['avg_rougeL'].values
best_i = np.argmax(rL_v)
clrs5 = ['#55A868' if i == best_i else '#4C72B0' for i in range(len(rL_v))]
ax5.bar(range(len(rL_v)), rL_v, color=clrs5, edgecolor='white')
ax5.set_xticks(range(len(rL_v)))
ax5.set_xticklabels(decode_df.index, rotation=30, ha='right', fontsize=7)
ax5.set_title('디코딩 전략별 ROUGE-L')
ax5.text(best_i, rL_v[best_i]+0.001, '★', ha='center', fontsize=12, color='gold')

# ⑥ Unique Ratio 비교
ax6 = fig.add_subplot(2, 3, 6)
ax6.bar(mn, [avg_base['unique_ratio'], avg_sft['unique_ratio']],
        color=['steelblue','tomato'], edgecolor='white')
ax6.set_title('생성 다양성 (Unique Token Ratio)'); ax6.set_ylabel('Ratio')

plt.tight_layout()
plt.savefig('/content/final_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('대시보드 저장 완료: /content/final_dashboard.png')

In [ ]:
# ── 최종 요약 출력 ────────────────────────────────────────────────────────────
print('='*65)
print('  최종 실험 결과 요약')
print('='*65)

print('\n[데이터셋]')
print(f'  원본 SFT          : {len(sft_raw):,}개')
print(f'  정제 후           : {len(cleaned_sft):,}개  (제거율 {(1-len(cleaned_sft)/len(sft_raw))*100:.1f}%)')
print(f'  KorQuAD 추가      : +{len(korquad_sft_sampled):,}개')
print(f'  최종 학습 데이터  : {len(augmented_sft):,}개')

print('\n[A] KoGPT2 기본 vs SFT 모델 (평균)')
for k in ['bleu4','rougeL','unique_ratio']:
    d   = avg_sft[k] - avg_base[k]
    pct = d / (avg_base[k] + 1e-8) * 100
    print(f'  {k:14s}: {avg_base[k]:.4f} → {avg_sft[k]:.4f}  ({pct:+.1f}%)')

print('\n[B] Reward Model')
print(f'  최종 Val Accuracy (chosen > rejected): {rm_vl_accs[-1]*100:.1f}%')
print(f'  (Random baseline = 50%)')

print('\n[C] 최적 디코딩 전략 (ROUGE-L 기준)')
best_d = decode_df['avg_rougeL'].idxmax()
print(f'  전략     : {best_d}')
print(f'  BLEU-4   : {decode_df.loc[best_d,"avg_bleu4"]:.4f}')
print(f'  ROUGE-L  : {decode_df.loc[best_d,"avg_rougeL"]:.4f}')

print('\n[D] SFT vs RM 스코어 선택 (평균)')
for k in ['bleu4','rougeL']:
    d   = avg_rm_sel[k] - avg_sft_g[k]
    pct = d / (avg_sft_g[k] + 1e-8) * 100
    print(f'  {k:10s}: {avg_sft_g[k]:.4f} → {avg_rm_sel[k]:.4f}  ({pct:+.1f}%)')

print('='*65)

---
## 📌 결론 및 향후 과제

### ✅ 달성 사항 (루브릭 대응)

| 루브릭 | 구현 내용 |
|---|---|
| ① 데이터 정제 + Generation 기법으로 성능 향상 | EDA → 클리닝 + Beam/Top-K/Top-P 비교 실험 |
| ② 새로운 데이터 수집·전처리 | KorQuAD 1.0 → SFT 형식 변환 후 증량 |
| ③ SFT/RM 학습 전략 적용 | SFT(3epoch) + RM Pairwise Loss 학습 |
| ④ SFT vs RM 정량/정성 비교 | Section 6-B: RM 스코어 기반 Best-of-N 선택 |
| ⑤ KoGPT2 기본 vs SFT 정량/정성 비교 | Section 6-A: BLEU/ROUGE/Unique Ratio 비교 |

### 🚀 향후 개선 방향
1. **더 큰 Foundation Model** — `skt/ko-gpt-trinity-1.2B-v0.5`
2. **LoRA 적용** — 메모리 효율적 파인튜닝으로 대형 모델 학습
3. **PPO 완성** — SFT + RM 이후 완전한 RLHF 파이프라인
4. **Human Feedback** — StackExchange 크롤링 등 실제 피드백 데이터
5. **Perplexity 평가** — 언어 모델링 품질 추가 정량화